In [40]:
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt

In [41]:
qubit_counts = [2,3,4,5,6,7,8,10,12,14,16,18,20,22,24,26]

n_layers = 2
n_trials = 200

In [42]:
def create_circuit(n_qubits, n_layers):

    dev = qml.device("lightning.qubit",wires = n_qubits, c_dtype=np.complex64)

    @qml.qnode(dev, diff_method="adjoint")
    def circuit(weights):
        for layer in range(n_layers):
            for q in range(n_qubits):
                qml.RY(weights[layer, q],wires=q)

            for q in range(n_qubits-1):
                qml.CNOT(wires=[q,q + 1])
                #qml.CNOT(wires=[q,np.random.uniform(0,n_qubits-1,size=None)])
                
        return qml.expval(qml.PauliZ(0))
    return circuit

In [43]:
def run_experiment(n_qubits, n_layers, n_trials):

    circuit = create_circuit(n_qubits,n_layers)
    
    all_gradients = []
    
    
    for trial in range(n_trials):
        weights = np.array(np.random.uniform(-np.pi,np.pi,size=(n_layers,n_qubits)),requires_grad=True)
        # old weights = np.random.uniform(-np.pi,np.pi,size=(n_layers,n_qubits),requires_grad=True)
        gradient = qml.grad(circuit)(weights)

        #new stuff added for single gradient calcs

        

        
        all_gradients.append(gradient)
        
    qml.draw_mpl(circuit,style='pennylane')(weights);
    
    return np.array(all_gradients)

In [44]:
results = []

for n_qubits in qubit_counts:

    print(f"Running {n_qubits} qubits...")

    gradients = run_experiment(n_qubits=n_qubits, n_layers=n_layers,n_trials=n_trials)
    parameter_var = np.var(gradients,axis=0)
    mean_var = np.mean(parameter_var)

    results.append({"Qubits": n_qubits,"Variance": mean_var})

    print(f"Mean grad variance: "f"{mean_var:.6e}")
    

Running 2 qubits...


WireError: Cannot run circuit(s) on lightning.qubit as they contain wires not found on the device: {0.19173434153633995, 0.60374978231413}

In [ ]:
qubits = np.array([r["Qubits"] for r in results])
variances = np.array([r["Variance"] for r in results])

plt.figure(figsize=(8,len(qubit_counts)))

plt.plot(qubits,variances,marker="o")
plt.yscale("log")

plt.xlabel("N Qubits")
plt.ylabel("Variance")

plt.grid(True)
plt.show()